In [84]:
import os
import re
import json
import copy
import random
import time
from typing import Any, Dict, List, Optional, Tuple

import requests
import numpy as np
import pandas as pd
from tqdm.auto import tqdm

seed = 1241
random.seed(seed)
np.random.seed(seed)


In [85]:
init_dataset = pd.read_json("hf://datasets/Team-ACE/ToolACE/data.json")
init_dataset.head(2)


,system,conversations
0,You are an expert in composing functions. You ...,"[{'from': 'user', 'value': 'I'm considering in..."
1,You are an expert in composing functions. You ...,"[{'from': 'user', 'value': 'Could you please f..."


## Parse ToolACE to (query, tool context, assistant output)


In [86]:
def tool_responses_to_text(responses: List[Dict[str, Any]]) -> str:
    parts = []
    for response in responses:
        parts.append(f"{response['name']}: {json.dumps(response['results'], ensure_ascii=False)}")
    return "\n".join(parts)


def parse_one_conversation(conv: List[Dict[str, str]]) -> List[Dict[str, str]]:
    res = []
    n = len(conv)
    i = 0

    while i < n:
        if conv[i].get('from') != 'tool':
            i += 1
            continue

        context = copy.deepcopy(json.loads(conv[i]['value']))
        entry = {}

        j = i
        while j >= 0 and conv[j].get('from') != 'user':
            j -= 1
        if j < 0:
            i += 1
            continue
        entry['query'] = conv[j]['value']

        j = i + 1
        while j < n:
            role = conv[j].get('from')
            if role == 'assistant':
                val = conv[j]['value']
                if val.startswith('[') and val.endswith(']'):
                    j += 1
                    continue
                break
            if role == 'tool':
                context += json.loads(conv[j]['value'])
                i = j
            j += 1

        if j >= n:
            i += 1
            continue

        entry['context'] = tool_responses_to_text(context)
        entry['output'] = conv[j]['value']
        res.append(entry)
        i += 1

    return res


def extract_tools_list_from_system(system_text: str) -> Optional[List[Dict[str, Any]]]:
    anchor = "Here is a list of functions"
    start_pos = system_text.find(anchor)
    s = system_text[start_pos:] if start_pos != -1 else system_text

    i0 = s.find("[")
    if i0 == -1:
        return None

    depth = 0
    in_str = False
    quote = ""
    esc = False
    start = None

    for i in range(i0, len(s)):
        ch = s[i]

        if in_str:
            if esc:
                esc = False
            elif ch == "\":
                esc = True
            elif ch == quote:
                in_str = False
            continue

        if ch in ('"', "'"):
            in_str = True
            quote = ch
            continue

        if ch == "[":
            if depth == 0:
                start = i
            depth += 1
        elif ch == "]":
            if depth > 0:
                depth -= 1
                if depth == 0 and start is not None:
                    block = s[start:i + 1]
                    try:
                        obj = json.loads(block)
                    except Exception:
                        return None
                    if isinstance(obj, list) and all(isinstance(x, dict) for x in obj):
                        return obj
                    return None
    return None


SyntaxError: unterminated string literal (detected at line 76) (1078687977.py, line 76)

In [ ]:
correct_dataset = []

for row in init_dataset.itertuples():
    parsed_conv = parse_one_conversation(row.conversations)
    parsed_tool_list = extract_tools_list_from_system(row.system)

    if parsed_tool_list and parsed_conv:
        tool_meta = [{k: t[k] for k in ('name', 'description') if k in t} for t in parsed_tool_list]
        for request in parsed_conv:
            request['context'] = f"{request['context']}\nAvailable tools: {json.dumps(tool_meta, ensure_ascii=False)}"

    correct_dataset.extend(parsed_conv)

len(correct_dataset)


## Generate tool-output contradictions via API


In [ ]:
OPENROUTER_API_KEY = 
if not OPENROUTER_API_KEY:
    raise ValueError("Set OPENROUTER_API_KEY before running")

OPENROUTER_MODEL_CANDIDATES = [
    "openai/gpt-4o-mini",
    "openrouter/free",
]

OPENROUTER_URL = "https://openrouter.ai/api/v1/chat/completions"
MAX_RETRIES_PER_MODEL = 3
BASE_BACKOFF_SECONDS = 0.8
HALLUCINATION_STRENGTH = 0.9

from concurrent.futures import ThreadPoolExecutor, as_completed


def _strength_level(strength: float) -> str:
    if strength <= 0.2:
        return "low"
    if strength <= 0.6:
        return "medium"
    return "high"


def _extract_json_object(text: str) -> Dict[str, Any]:
    text = text.strip()
    try:
        return json.loads(text)
    except Exception:
        pass

    m = re.search(r"\{[\s\S]*\}", text)
    if not m:
        raise ValueError("No JSON object found in model output")
    return json.loads(m.group(0))


def _build_prompt(query: str, tool_context: str, output: str, strength: float) -> str:
    level = _strength_level(strength)
    return f"""
Return strict JSON only with keys:
- rewritten_answer: string
- changed_facts: array of strings

Task:
Rewrite the assistant answer so it contradicts factual details from TOOL OUTPUT.
You must change facts that are explicitly present in the original assistant answer and grounded in the tool output.
Do not rewrite style; keep answer natural and close to original wording.

Rules:
- Contradiction strength: {strength} ({level})
- low: change exactly 1 grounded fact
- medium: change 1-2 grounded facts
- high: change 2+ grounded facts
- Keep language and structure similar.
- Do not mention that you are contradicting or hallucinating.
- Do not add disclaimers.

User query:
{query}

TOOL OUTPUT:
{tool_context}

Original assistant answer:
{output}
""".strip()


def mutate_answer_via_api(entry: Dict[str, Any], strength: float = HALLUCINATION_STRENGTH) -> Tuple[str, List[Dict[str, Any]], Dict[str, Any]]:
    prompt = _build_prompt(entry['query'], entry['context'], entry['output'], strength)

    headers = {
        "Authorization": f"Bearer {OPENROUTER_API_KEY}",
        "Content-Type": "application/json",
    }

    last_error = None
    for model in OPENROUTER_MODEL_CANDIDATES:
        for attempt in range(1, MAX_RETRIES_PER_MODEL + 1):
            payload = {
                "model": model,
                "messages": [
                    {
                        "role": "system",
                        "content": "You are a data generation assistant. Output must be strict JSON object only.",
                    },
                    {"role": "user", "content": prompt},
                ],
                "temperature": 0.6,
            }

            try:
                resp = requests.post(OPENROUTER_URL, headers=headers, json=payload, timeout=120)
                if resp.status_code >= 400:
                    raise RuntimeError(f"{resp.status_code} {resp.text[:500]}")

                data = resp.json()
                content = data["choices"][0]["message"]["content"]
                parsed = _extract_json_object(content)

                rewritten = str(parsed.get("rewritten_answer", "")).strip()
                changed_facts = parsed.get("changed_facts", [])
                if not isinstance(changed_facts, list):
                    changed_facts = [str(changed_facts)]

                if not rewritten:
                    raise ValueError("Empty rewritten_answer")
                if rewritten == entry['output']:
                    raise ValueError("Model returned unchanged answer")

                labels = []
                for fact in changed_facts:
                    if not isinstance(fact, str) or not fact.strip():
                        continue
                    labels.append({
                        "type": "tool_output_contradiction",
                        "kind": "api_changed_fact",
                        "text": fact.strip(),
                    })

                meta = {
                    "status": "ok",
                    "strength": strength,
                    "mutation_direction": "answer_changed_context_kept",
                    "model": model,
                    "changed_facts": [x.get("text") for x in labels],
                }
                return rewritten, labels, meta

            except Exception as e:
                last_error = f"model={model}, attempt={attempt}: {e}"
                if attempt < MAX_RETRIES_PER_MODEL:
                    time.sleep(BASE_BACKOFF_SECONDS * attempt)
                continue

    raise RuntimeError(last_error or "All model attempts failed")


def _process_entry(item: Dict[str, Any], p: float, strength: float, fail_mode: str) -> Dict[str, Any]:
    entry = copy.deepcopy(item)

    try:
        if np.random.uniform(0, 1) < p:
            old_output = entry['output']
            old_context = entry['context']

            new_output, labels, meta = mutate_answer_via_api(entry, strength=strength)

            entry['original_output'] = old_output
            entry['original_context'] = old_context
            entry['output'] = new_output
            entry['context'] = old_context
            entry['hallucination_labels'] = labels
            entry['meta'] = meta
        else:
            entry['original_output'] = entry['output']
            entry['original_context'] = entry['context']
            entry['hallucination_labels'] = []
            entry['meta'] = {"status": "clean"}

    except Exception as e:
        if fail_mode == 'raise':
            raise
        entry['original_output'] = entry.get('output', '')
        entry['original_context'] = entry.get('context', '')
        entry['hallucination_labels'] = []
        entry['meta'] = {
            "status": "api_failed",
            "error": str(e),
            "strength": strength,
        }

    return entry


def corrupt(dataset: List[Dict[str, Any]], p: float = 0.5, start: int = 0, end: Optional[int] = None, fail_mode: str = "skip", strength: float = HALLUCINATION_STRENGTH, num_workers: int = 8) -> List[Dict[str, Any]]:
    if end is None:
        end = len(dataset)

    items = dataset[start:end]
    if num_workers <= 1:
        return [_process_entry(item, p, strength, fail_mode) for item in tqdm(items, total=len(items))]

    results: List[Optional[Dict[str, Any]]] = [None] * len(items)

    with ThreadPoolExecutor(max_workers=num_workers) as ex:
        futures = {ex.submit(_process_entry, item, p, strength, fail_mode): idx for idx, item in enumerate(items)}
        for fut in tqdm(as_completed(futures), total=len(futures)):
            idx = futures[fut]
            results[idx] = fut.result()

    return results


In [ ]:
hallucinated_dataset = corrupt(
    correct_dataset,
    p=1.0,
    start=0,
    end=len(correct_dataset),
    fail_mode="skip",
    strength=0.9,
    num_workers=2,  # increase to 12-16 if your rate limits allow it
)

from collections import Counter
Counter(item.get("meta", {}).get("status", "unknown") for item in hallucinated_dataset)


  0%|          | 2/1034 [00:12<1:44:29,  6.07s/it]


In [ ]:
with open("tool_output_contradiction_dataset.jsonl", "w", encoding="utf-8") as f:
    for sample in hallucinated_dataset:
        f.write(json.dumps(sample, ensure_ascii=False) + "\n")

print("Saved to tool_output_contradiction_dataset.jsonl")


## Quick viewer


In [ ]:
import re
import html
from difflib import SequenceMatcher
from IPython.display import HTML, display


def _tokenize(text: str):
    return re.findall(r"\w+|[^\w\s]", text, flags=re.UNICODE)


def _join_tokens(tokens):
    s = " ".join(tokens)
    s = re.sub(r"\s+([,.:;!?%)\]\}])", r"\1", s)
    s = re.sub(r"([([\{])\s+", r"\1", s)
    return s


def _highlight_diff(old, new):
    old_toks = _tokenize(old)
    new_toks = _tokenize(new)
    sm = SequenceMatcher(None, old_toks, new_toks)
    old_html, new_html = [], []

    for tag, i1, i2, j1, j2 in sm.get_opcodes():
        a = html.escape(_join_tokens(old_toks[i1:i2]))
        b = html.escape(_join_tokens(new_toks[j1:j2]))
        if tag == "equal":
            old_html.append(a)
            new_html.append(b)
        elif tag == "delete":
            old_html.append(f'<span style="background:#ffd6d6;text-decoration:line-through;">{a}</span>')
        elif tag == "insert":
            new_html.append(f'<span style="background:#d6ffd6;">{b}</span>')
        else:
            old_html.append(f'<span style="background:#ffd6d6;text-decoration:line-through;">{a}</span>')
            new_html.append(f'<span style="background:#d6ffd6;">{b}</span>')

    return ''.join(old_html), ''.join(new_html)


def show_dialogue(idx, dataset=None):
    ds = dataset if dataset is not None else hallucinated_dataset
    row = ds[idx]

    q = html.escape(row.get("query", ""))
    old_ctx = row.get("original_context", row.get("context", ""))
    new_ctx = row.get("context", "")
    old_out = row.get("original_output", row.get("output", ""))
    new_out = row.get("output", "")
    meta = row.get("meta", {})

    old_ctx_h, new_ctx_h = _highlight_diff(old_ctx, new_ctx)
    old_out_h, new_out_h = _highlight_diff(old_out, new_out)

    html_block = f"""
    <div style='font-family: ui-monospace, SFMono-Regular, Menlo, monospace; line-height:1.45;'>
      <h3>Dialogue #{idx}</h3>
      <p><b>Query:</b> {q}</p>
      <p><b>Status:</b> {html.escape(str(meta.get('status','')))}</p>
      <p><b>Strength:</b> {html.escape(str(meta.get('strength','')))}</p>
      <p><b>Model:</b> {html.escape(str(meta.get('model','')))}</p>
      <hr>
      <h4>Context (before)</h4>
      <div style='white-space:pre-wrap;border:1px solid #ddd;padding:10px;border-radius:8px'>{old_ctx_h}</div>
      <h4>Context (after)</h4>
      <div style='white-space:pre-wrap;border:1px solid #ddd;padding:10px;border-radius:8px'>{new_ctx_h}</div>
      <h4>Answer (before)</h4>
      <div style='white-space:pre-wrap;border:1px solid #ddd;padding:10px;border-radius:8px'>{old_out_h}</div>
      <h4>Answer (after)</h4>
      <div style='white-space:pre-wrap;border:1px solid #ddd;padding:10px;border-radius:8px'>{new_out_h}</div>
    </div>
    """
    display(HTML(html_block))


In [ ]:
# Example usage:
# show_dialogue(45)
